# Notebook 01: Community Detection
## Finding Structure in the Knowledge Graph

Notebook 00 gave us a **Knowledge Graph** — a network of entities (Qatar, Messi, Lusail Stadium, …)
connected by relationships. But the graph is still flat: every node sits at the same level.
To answer *global* questions like *"What are the main themes of the 2022 World Cup?"* we need
to identify higher-level structure.

**Community detection** partitions the graph into tightly-knit clusters — groups of nodes that
have more connections to each other than to the rest of the graph. In GraphRAG each community
becomes an independent unit of summarization (Notebook 02), and the resulting summaries feed
the Map-Reduce query (Notebook 03).

**Paper reference:** Edge et al. (2025) — *From Local to Global: A GraphRAG Approach to Query-Focused Summarization*,
§3.1.4 and Appendix B.

### What this notebook does

```
data/output/wc2022_trimmed_graph.json
           │
           ▼  § 3.1.4  Leiden community detection (2 resolution levels)
     C0 — coarse communities  (fewer, larger)
     C1 — fine   communities  (more, smaller) ← used by Notebook 02
           │
           ▼
data/output/wc2022_trimmed_communities.json
```

## Imports and Setup

In [ ]:
import json
from pathlib import Path

import networkx as nx
import igraph as ig
import leidenalg
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

Path("../data/output").mkdir(parents=True, exist_ok=True)
print("Imports OK")

In [ ]:
# Change PDF_STEM here if you re-ran notebook 00 on a different file
PDF_STEM = "wc2022_trimmed"

GRAPH_PATH       = Path("../data/output") / f"{PDF_STEM}_graph.json"
COMMUNITIES_PATH = Path("../data/output") / f"{PDF_STEM}_communities.json"

print(f"Input : {GRAPH_PATH}")
print(f"Output: {COMMUNITIES_PATH}")

---
## Step 1 — Load the Graph  *(§ 3.1.4)*

We reload the `networkx` graph that notebook 00 saved. We then drop the **27 isolated nodes**
(nodes with no edges). Isolated nodes are extraction noise — the LLM named an entity but found
no relationship for it. They cannot form meaningful communities and would each become a
singleton cluster in the Leiden output.

In [ ]:
with open(GRAPH_PATH, encoding="utf-8") as f:
    G = nx.node_link_graph(json.load(f))

print(f"Full graph : {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Drop isolates
G_sub = G.subgraph([n for n, d in G.degree() if d > 0]).copy()
print(f"Connected  : {G_sub.number_of_nodes()} nodes, {G_sub.number_of_edges()} edges")
print(f"Dropped    : {G.number_of_nodes() - G_sub.number_of_nodes()} isolated nodes")

top = sorted(G_sub.degree(), key=lambda x: x[1], reverse=True)[:8]
print("\nTop-8 by degree:")
for name, deg in top:
    print(f"  {name:<35} {deg:>3}  ({G_sub.nodes[name].get('type', '?')})")

---
## Step 2 — Convert to igraph  *(§ 3.1.4)*

`leidenalg` operates on **igraph** graph objects, not networkx ones. The conversion is
lossless: all node names and edge weights are preserved via vertex/edge attributes.

> **Why Leiden and not Louvain?**  
> The paper explicitly chooses Leiden (§3.1.4, citing Traag et al. 2019) because Louvain
> can produce communities with poor internal connectivity — so-called "badly connected"
> communities. Leiden adds a refinement step that guarantees well-connected partitions.

In [ ]:
# igraph.Graph.from_networkx preserves node order: igraph vertex i == node_names[i]
node_names = list(G_sub.nodes())
g_ig = ig.Graph.from_networkx(G_sub)

print(f"igraph: {g_ig.vcount()} vertices, {g_ig.ecount()} edges")
print(f"Edge attributes: {g_ig.edge_attributes()}")

# Use edge weights if they exist
weights = "weight" if "weight" in g_ig.edge_attributes() else None
print(f"Using weights: {weights}")

---
## Step 3 — Leiden Community Detection  *(§ 3.1.4, Appendix B)*

The paper produces a **hierarchical** partition by running the algorithm at two resolution
levels, mirroring Figure 4:

| Level | Partition type | Resolution | Communities |
|-------|---------------|------------|-------------|
| **C0** | `ModularityVertexPartition` | — (default ≈ 1.0) | Fewer, larger |
| **C1** | `RBConfigurationVertexPartition` | 1.5 | More, smaller |

**C1 is the leaf level** — these are the communities that get summarized in Notebook 02.
C0 is the coarser root level; it is included here to mirror the paper's Figure 4 and to
show students how resolution controls granularity.

> **Resolution parameter:** In `RBConfigurationVertexPartition`, higher values produce
> finer-grained communities. We use 1.5 for C1 to obtain a clearly larger number of
> communities than C0 on this dataset (12 → 15). The paper leaves the exact value
> dataset-dependent (Appendix B).

We fix `seed=42` so that runs are reproducible — Leiden is a randomised algorithm.

In [ ]:
SEED = 42

print("Running Leiden C0 (ModularityVertexPartition) ...")
partition_c0 = leidenalg.find_partition(
    g_ig,
    leidenalg.ModularityVertexPartition,
    weights=weights,
    seed=SEED,
)
print(f"  C0: {len(partition_c0)} communities")

print("Running Leiden C1 (RBConfigurationVertexPartition, resolution=1.5) ...")
partition_c1 = leidenalg.find_partition(
    g_ig,
    leidenalg.RBConfigurationVertexPartition,
    resolution_parameter=1.5,
    weights=weights,
    seed=SEED,
)
print(f"  C1: {len(partition_c1)} communities")

In [ ]:
def community_summary(partition, node_names, label):
    print(f"\n{label} — {len(partition)} communities:")
    for i, members in enumerate(partition):
        names = [node_names[v] for v in members]
        preview = ", ".join(names[:5])
        suffix = f" … (+{len(names)-5} more)" if len(names) > 5 else ""
        print(f"  [{i:02d}] {len(names):>3} nodes — {preview}{suffix}")

community_summary(partition_c0, node_names, "C0 (coarse)")
community_summary(partition_c1, node_names, "C1 (fine)")

---
## Step 4 — Save `communities.json`

We store both levels under the keys `"C0"` and `"C1"`. Each level maps a community ID
(string) to the list of node names it contains. Notebook 02 will read `"C1"` to generate
one LLM summary per leaf community.

In [ ]:
def partition_to_dict(partition, node_names):
    return {
        str(comm_id): [node_names[v] for v in members]
        for comm_id, members in enumerate(partition)
    }

communities = {
    "C0": partition_to_dict(partition_c0, node_names),
    "C1": partition_to_dict(partition_c1, node_names),
}

with open(COMMUNITIES_PATH, "w", encoding="utf-8") as f:
    json.dump(communities, f, ensure_ascii=False, indent=2)

print(f"Saved {COMMUNITIES_PATH}")
print(f"File size: {COMMUNITIES_PATH.stat().st_size / 1024:.1f} KB")
print(f"\nC0: {len(communities['C0'])} communities")
print(f"C1: {len(communities['C1'])} communities  ← Notebook 02 uses these")

---
## Visualisation — Replicating Figure 4

The paper's Figure 4 shows the graph partitioned at two resolution levels side by side.
We reproduce this: both plots share the same spring-layout positions so you can directly
compare how C0 and C1 differ. Nodes are sized by degree; colours indicate community
membership.

In [ ]:
# Build membership lookup: node_name -> community_id for each level
membership_c0 = {node_names[v]: cid for cid, members in enumerate(partition_c0) for v in members}
membership_c1 = {node_names[v]: cid for cid, members in enumerate(partition_c1) for v in members}

# Fixed layout — computed once, shared by both subplots
pos = nx.spring_layout(G_sub, seed=42, k=1.2)

degrees   = dict(G_sub.degree())
node_list = list(G_sub.nodes())
sizes     = [50 + degrees[n] * 30 for n in node_list]

def make_color_list(membership, node_list, n_communities):
    cmap = cm.get_cmap("tab20", max(n_communities, 1))
    return [cmap(membership[n] / max(n_communities - 1, 1)) for n in node_list]

n_c0 = len(partition_c0)
n_c1 = len(partition_c1)
colors_c0 = make_color_list(membership_c0, node_list, n_c0)
colors_c1 = make_color_list(membership_c1, node_list, n_c1)

fig, axes = plt.subplots(1, 2, figsize=(18, 8), facecolor="#1a1a2e")

for ax, colors, membership, n_comm, label in [
    (axes[0], colors_c0, membership_c0, n_c0, f"C0 — {n_c0} communities (coarse)"),
    (axes[1], colors_c1, membership_c1, n_c1, f"C1 — {n_c1} communities (fine)"),
]:
    ax.set_facecolor("#1a1a2e")
    nx.draw_networkx_edges(
        G_sub, pos, ax=ax,
        edge_color="#444466", alpha=0.5, width=0.8,
    )
    nx.draw_networkx_nodes(
        G_sub, pos, ax=ax,
        nodelist=node_list,
        node_color=colors,
        node_size=sizes,
        alpha=0.9,
    )
    # Label only the highest-degree nodes to avoid clutter
    top_nodes = {n for n, d in sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:15]}
    labels    = {n: n for n in top_nodes}
    nx.draw_networkx_labels(
        G_sub, pos, labels=labels, ax=ax,
        font_size=7, font_color="white",
    )
    ax.set_title(label, color="white", fontsize=13, pad=10)
    ax.axis("off")

plt.suptitle(
    "GraphRAG — Leiden Community Hierarchy (2022 FIFA World Cup)",
    color="white", fontsize=15, y=1.01,
)
plt.tight_layout()
plt.savefig(
    Path("../data/output") / f"{PDF_STEM}_communities.png",
    dpi=150, bbox_inches="tight", facecolor="#1a1a2e",
)
plt.show()
print("Figure saved.")

---
## Summary

| Paper section | What we did |
|---|---|
| § 3.1.4 | Chose Leiden (not Louvain) for well-connected partition guarantee |
| § 3.1.4 | Ran two resolution levels — C0 (coarse) and C1 (fine) |
| App. B | Used `ModularityVertexPartition` for C0, `RBConfigurationVertexPartition` for C1 |
| Figure 4 | Two-level visualisation with shared layout |

The communities are saved at `data/output/wc2022_trimmed_communities.json`.

**Next → Notebook 02:** For each C1 community, we ask the LLM to write a structured JSON
summary report — capturing the theme, key findings, and an importance rating (§3.1.5, Appendix E.2).